In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')  

 **Air Quality Data Exploration and Preprocessing for Asthma Risk Prediction**

This notebook analyzes hourly air quality data to identify environmental triggers for asthmatic patients. We focus on PM values (PM2.5, PM10), temperature, humidity, pressure and air quality index to generate predictions and alert thresholds.

In [2]:
# Load air quality dataset
airquality_dataset = pd.read_csv('../data/raw/AirQualityData.csv')
print("Dataset shape:", airquality_dataset.shape)
print("\nColumn names:")
print(airquality_dataset.columns.tolist())

Dataset shape: (4000, 23)

Column names:
['Date', 'Time', 'CO(GT)', 'NOx(GT)', 'NO2(GT)', 'O3(GT)', 'SO2(GT)', 'PM2.5', 'PM10', 'Temperature', 'Humidity', 'Pressure', 'WindSpeed', 'WindDirection', 'CO_NOx_Ratio', 'NOx_NO2_Ratio', 'Temp_Humidity_Index', 'AirQualityIndex', 'CO_MA3', 'NO2_MA3', 'O3_MA3', 'DayOfWeek', 'Hour']


In [3]:
# Table for air quality dataset core features
core_features_info = {
    "Feature": ["PM2.5", "PM10", "Temperature", "Humidity", "Pressure", "AirQualityIndex"],
    "Unit": ["Î¼g/mÂ³", "Î¼g/mÂ³", "Â°C", "%", "hPa", "index"],
    "Meaning": [
        "Fine particulate matter â‰¤2.5 micrometers - major asthma trigger",
        "Coarse particulate matter â‰¤10 micrometers - respiratory irritant",
        "Ambient air temperature",
        "Relative humidity level",
        "Atmospheric pressure",
        "Composite air quality metric (0-500 scale)"
    ],
    "Expected Range": [
        "0-500", "0-1000", "-10 to +50", "0-100", "950-1050", "0-500"
    ]
}
core_features_table = pd.DataFrame(core_features_info)
print("Core Features Summary:")
display(core_features_table)

Core Features Summary:


,Feature,Unit,Meaning,Expected Range
0,PM2.5,Î¼g/mÂ³,Fine particulate matter â‰¤2.5 micrometers - m...,0-500
1,PM10,Î¼g/mÂ³,Coarse particulate matter â‰¤10 micrometers - ...,0-1000
2,Temperature,Â°C,Ambient air temperature,-10 to +50
3,Humidity,%,Relative humidity level,0-100
4,Pressure,hPa,Atmospheric pressure,950-1050
5,AirQualityIndex,index,Composite air quality metric (0-500 scale),0-500


In [4]:
# Table for temporal features
temporal_info = {
    "Feature": ["Date", "Time", "DayOfWeek", "Hour"],
    "Unit": ["YYYY-MM-DD", "HH:MM", "0-6 (0=Monday)", "0-23"],
    "Meaning": [
        "Calendar date of measurement",
        "Time of day (24-hour format)",
        "Day of the week",
        "Hour of day"
    ],
    "Usage": [
        "Timestamp reference",
        "Hourly time reference",
        "Identify weekday vs weekend patterns",
        "Identify peak pollution hours"
    ]
}
temporal_table = pd.DataFrame(temporal_info)
print("Temporal Features Summary:")
display(temporal_table)

Temporal Features Summary:


,Feature,Unit,Meaning,Usage
0,Date,YYYY-MM-DD,Calendar date of measurement,Timestamp reference
1,Time,HH:MM,Time of day (24-hour format),Hourly time reference
2,DayOfWeek,0-6 (0=Monday),Day of the week,Identify weekday vs weekend patterns
3,Hour,0-23,Hour of day,Identify peak pollution hours


In [5]:
# Display first rows of the dataset
print("First 10 rows of Air Quality Dataset:")
display(airquality_dataset.head(10))

First 10 rows of Air Quality Dataset:


,Date,Time,CO(GT),NOx(GT),NO2(GT),O3(GT),SO2(GT),PM2.5,PM10,Temperature,...,WindDirection,CO_NOx_Ratio,NOx_NO2_Ratio,Temp_Humidity_Index,AirQualityIndex,CO_MA3,NO2_MA3,O3_MA3,DayOfWeek,Hour
0,2024-01-01,00:00,3.807947,172.026768,144.333317,118.120832,1.215679,147.349671,208.803124,28.564580,...,209.984267,0.022008,1.183671,3.541778,343.353046,3.807947,144.333317,118.120832,0,0
1,2024-01-01,01:00,9.512072,241.824266,137.769318,15.325830,1.016178,40.979839,145.595579,6.793192,...,319.534890,0.039173,1.742635,0.727989,206.282028,6.660009,141.051317,66.723331,0,1
2,2024-01-01,02:00,7.346740,228.288118,20.055086,44.377036,24.140910,72.594740,26.155000,24.436552,...,274.644300,0.032042,10.842422,7.378322,140.170920,6.888920,100.719240,59.274566,0,2
3,2024-01-01,03:00,6.026719,47.016072,184.591909,139.488603,2.435392,134.339724,276.367944,26.463951,...,312.266023,0.125515,0.253330,21.684266,307.928588,7.628510,114.138771,66.397156,0,3
4,2024-01-01,04:00,1.644585,45.625591,114.125968,95.634768,48.752095,99.007422,294.295449,10.530331,...,21.392120,0.035272,0.396310,9.627596,370.134556,5.006015,106.257654,93.166802,0,4
5,2024-01-01,05:00,1.644346,81.184136,73.381379,167.106462,11.881194,149.021353,135.002590,16.371354,...,114.525516,0.020008,1.091458,7.409536,127.292467,3.105216,124.033085,134.076611,0,5
6,2024-01-01,06:00,0.675028,108.961343,151.551178,77.746502,24.160572,192.435541,187.961634,13.169519,...,114.736300,0.006139,0.714261,7.992301,149.699951,1.321319,113.019508,113.495911,0,6
7,2024-01-01,07:00,8.675144,123.128219,52.215726,156.696413,3.995744,27.726604,140.586602,37.630705,...,236.536221,0.069889,2.313756,27.675402,320.146958,3.664839,92.382761,133.849793,0,7
8,2024-01-01,08:00,6.051039,204.229468,139.008458,81.363023,26.580372,143.365705,103.345099,9.397699,...,161.501418,0.029484,1.458694,8.999641,482.611630,5.133737,114.258454,105.268646,0,8
9,2024-01-01,09:00,7.109919,17.947449,8.902530,176.285442,19.617850,49.917110,32.422450,3.249537,...,279.890137,0.375244,1.812410,2.994900,106.935768,7.278700,66.708905,138.114960,0,9


In [6]:
# Dataset information and data types
print("\nAir Quality Dataset Info:")
airquality_dataset.info()


Air Quality Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Date                 4000 non-null   str    
 1   Time                 4000 non-null   str    
 2   CO(GT)               4000 non-null   float64
 3   NOx(GT)              4000 non-null   float64
 4   NO2(GT)              4000 non-null   float64
 5   O3(GT)               4000 non-null   float64
 6   SO2(GT)              4000 non-null   float64
 7   PM2.5                4000 non-null   float64
 8   PM10                 4000 non-null   float64
 9   Temperature          4000 non-null   float64
 10  Humidity             4000 non-null   float64
 11  Pressure             4000 non-null   float64
 12  WindSpeed            4000 non-null   float64
 13  WindDirection        4000 non-null   float64
 14  CO_NOx_Ratio         4000 non-null   float64
 15  NOx_NO2_Ratio        4

In [11]:
# Check for duplicates in air quality dataset
duplicates = airquality_dataset[airquality_dataset.duplicated(keep=False)]
print(f"Number of duplicate rows found: {len(duplicates)}")
if len(duplicates) > 0:
    duplicates_sorted = duplicates.sort_values(list(duplicates.columns))
    print("\nDuplicate rows:")
    display(duplicates_sorted.head(20))
else:
    print("\nNo duplicate rows found")

# Remove any duplicates
airquality_dataset.drop_duplicates(inplace=True)
print(f"\nDataset shape after removing duplicates: {airquality_dataset.shape}")

Number of duplicate rows found: 0

No duplicate rows found

Dataset shape after removing duplicates: (4000, 23)


In [12]:
# Check for missing values
print("Missing values per column:")
missing_values = airquality_dataset.isnull().sum()
print(missing_values)
print(f"\nTotal missing values: {missing_values.sum()}")

# Check for negative or impossible values in key features
anomalies = []
if (airquality_dataset['PM2.5'] < 0).any():
    anomalies.append("Negative PM2.5 values detected")
if (airquality_dataset['PM10'] < 0).any():
    anomalies.append("Negative PM10 values detected")
if (airquality_dataset['Humidity'] < 0).any() or (airquality_dataset['Humidity'] > 100).any():
    anomalies.append("Humidity outside 0-100% range detected")
if (airquality_dataset['Pressure'] < 900).any() or (airquality_dataset['Pressure'] > 1100).any():
    anomalies.append("Pressure outside typical range (900-1100) detected")

if anomalies:
    for anomaly in anomalies:
        print(f"{anomaly}")
else:
    print("No anomalous values detected in core features")

Missing values per column:
Date                   0
Time                   0
CO(GT)                 0
NOx(GT)                0
NO2(GT)                0
O3(GT)                 0
SO2(GT)                0
PM2.5                  0
PM10                   0
Temperature            0
Humidity               0
Pressure               0
WindSpeed              0
WindDirection          0
CO_NOx_Ratio           0
NOx_NO2_Ratio          0
Temp_Humidity_Index    0
AirQualityIndex        0
CO_MA3                 0
NO2_MA3                0
O3_MA3                 0
DayOfWeek              0
Hour                   0
dtype: int64

Total missing values: 0
No anomalous values detected in core features


In [13]:
# Descriptive statistics for all numerical columns
print("\nDescriptive Statistics")
print(airquality_dataset.describe())


Descriptive Statistics
            CO(GT)      NOx(GT)      NO2(GT)       O3(GT)      SO2(GT)  \
count  4000.000000  4000.000000  4000.000000  4000.000000  4000.000000   
mean      5.025385   148.126633   100.213189    89.914815    26.081045   
std       2.874632    85.999247    57.074947    52.003484    14.059684   
min       0.100115     1.009185     1.010513     1.055442     1.012370   
25%       2.514242    73.636615    51.326622    44.179487    14.220565   
50%       5.054973   146.440690    99.508855    88.956924    26.321359   
75%       7.524652   221.823697   149.666167   136.333683    37.833728   
max       9.997205   299.838744   199.934968   179.986544    49.993700   

             PM2.5         PM10  Temperature     Humidity     Pressure  ...  \
count  4000.000000  4000.000000  4000.000000  4000.000000  4000.000000  ...   
mean    104.765999   153.591417    17.305228    54.626284   999.862679  ...   
std      56.344868    83.080911    12.943632    25.844003    28.897118  

In [3]:
print(airquality_dataset.dtypes)

Date                       str
Time                       str
CO(GT)                 float64
NOx(GT)                float64
NO2(GT)                float64
O3(GT)                 float64
SO2(GT)                float64
PM2.5                  float64
PM10                   float64
Temperature            float64
Humidity               float64
Pressure               float64
WindSpeed              float64
WindDirection          float64
CO_NOx_Ratio           float64
NOx_NO2_Ratio          float64
Temp_Humidity_Index    float64
AirQualityIndex        float64
CO_MA3                 float64
NO2_MA3                float64
O3_MA3                 float64
DayOfWeek                int64
Hour                     int64
dtype: object


In [4]:
# Define AQI category target from AirQualityIndex
bins = [-1, 50, 100, 150, 200, 300, 500]
labels = ["Good", "Moderate", "Unhealthy_SG", "Unhealthy", "Very_Unhealthy", "Hazardous"]

airquality_dataset["AQI_Category"] = pd.cut(
    airquality_dataset["AirQualityIndex"],
    bins=bins,
    labels=labels
)

print("Air Quality Dataset Class Balance (AQI_Category):")
print(airquality_dataset["AQI_Category"].value_counts())
print(airquality_dataset["AQI_Category"].value_counts(normalize=True))

Air Quality Dataset Class Balance (AQI_Category):
AQI_Category
Hazardous         1619
Very_Unhealthy     788
Unhealthy_SG       409
Moderate           404
Unhealthy          395
Good               385
Name: count, dtype: int64
AQI_Category
Hazardous         0.40475
Very_Unhealthy    0.19700
Unhealthy_SG      0.10225
Moderate          0.10100
Unhealthy         0.09875
Good              0.09625
Name: proportion, dtype: float64


In [5]:
airquality_dataset.to_csv('../data/processed/AirQualityProcessed', index=False)